# “做官”专科模型：Qwen2.5-7B QLoRA 微调（Kaggle T4 x2）

目标：把 Qwen2.5-7B 微调成懂中国官制史料的专科模型，训完导出 GGUF（Q4_K_M，约 4GB），本地可用 Ollama / LM Studio 跑。

**先跑通，再放大**：本笔记本默认读 `sample-qa.jsonl`（3 条样本），先确认全流程能跑、能合并、能导出 GGUF，再把 `DATA_PATH` 换成你的真实问答对文件。

额度（2026-11-08 核实）：Kaggle GPU 每周 30 小时、TPU 20 小时，本周全剩；单次会话最长 12 小时，30 小时要分 3 次跑，靠检查点续训。

In [ ]:
# 1. 安装（Kaggle Notebook 里跑一次即可）
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. 配置：只改这里
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # 4-bit 底座，QLoRA 用
DATA_PATH  = "sample-qa.jsonl"   # 换成你的问答对文件，如 "qa.jsonl"
OUTPUT_DIR = "outputs/officialdom-qwen2.5-7b"  # LoRA 适配器与检查点
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16
EPOCHS = 2            # 首轮先跑 2 轮看效果，再决定加不加
LEARNING_RATE = 2e-4
BATCH_SIZE = 2        # T4 x2 上可试 4；爆显存就降回 2
GRAD_ACCUM = 4

In [ ]:
# 3. 训练数据：3 条样本直接写进笔记本（冒烟测试用，不读外部文件）
import json

examples = [
  {
    "instruction": "清代军机处和明代内阁有什么区别？",
    "input": "",
    "output": "核心区别在名分与权力来源。明代内阁是皇帝的秘书班子，阁臣称大学士，品级不高、没有法定行政权，票拟须经皇帝朱批才生效，权力大小全看皇帝信任。清代军机处同样是皇帝的秘书机构，但军机大臣多由亲王、尚书兼任，品级高、直通皇帝，承旨办理军国大事、传达谕旨，权力更集中；同时它也没有独立的法定决策权，仍是皇帝的办事机构。简言之：两者都是皇权的延伸而非独立的决策机关，军机处比内阁更贴近皇帝、办事效率更高，但制度属性没有本质变化。"
  },
  {
    "instruction": "清代官员上奏折和上题本有什么不同？",
    "input": "",
    "output": "奏折与题本是清代两种并行的上行文书。题本是正式公文，经通政使司、内阁层层转发，内容公开、程式固定，用于常规政务。奏折由官员直接密呈皇帝，封口加“请安折”或密折，皇帝朱批后发回本人，外人不得与闻，用于机密要事与个人建言。雍正朝以后奏折制度推广，皇帝借此绕过官僚层级直接掌握地方实情，这是皇权强化信息控制的手段。"
  },
  {
    "instruction": "科举出身的官员和捐纳出身的官员在仕途上有什么实际差别？",
    "input": "",
    "output": "差别主要在起点、升迁速度与同僚评价。科举正途出身者授官起点高，翰林院庶吉士、进士馆选等路径通向清要之职，升迁较快，也更容易获得“正途”声望。捐纳出身者需先以银钱换得官职或资格，授官多为地方佐贰、冷僻之缺，升迁受限，且常被正途官员轻视。但捐纳在清代中后期是财政收入的重要来源，实际任职者众多，制度上对两者最终都承认官身，差别更多体现在仕途天花板与声誉上，而非能否做官。"
  }
]

print(f"共 {len(examples)} 条")
print(examples[0])

def to_text(ex):
    user = ex["instruction"] + ("\n" + ex["input"] if ex.get("input") else "")
    return (
        "<|im_start|>system\n你是一位精通中国历代官制的学者，回答要准确、克制，有出处意识，不确定的地方明说不确定。<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{ex['output']}<|im_end|>"
    )

texts = [to_text(ex) for ex in examples]


In [ ]:
# 4. 加载底座 + 挂 LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 5. 训练
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = Dataset.from_dict({"text": texts})

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        save_steps=200,          # 断点续训靠它：会话断了从最新检查点恢复
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)
trainer.train()
# 断点续训（会话中断后重跑本节时用）：trainer.train(resume_from_checkpoint=True)

In [ ]:
# 6. 合并 LoRA 回底座，并导出 GGUF（Q4_K_M，约 4GB）
#    注意：合并 + 导出占内存与时间，样本量小时很快；大数据量时留足会话时间。
model.save_pretrained_merged(
    "outputs/officialdom-qwen2.5-7b-merged",
    tokenizer,
    save_method="merged_16bit",
)
model.save_pretrained_gguf(
    "outputs/officialdom-qwen2.5-7b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("完成：merged 模型与 GGUF 已写入 outputs/，从 Kaggle 右侧 Output 里下载 GGUF 文件即可。")

## 跑完之后

- **先验效果**：拿 10 个你自己知道答案的官制问题考它，看行话、文体对不对，史实细节有没有编——编了是正常的，微调不保证史实。
- **史实要准**：下一步是加 RAG——史料切块入库，问答时先检索原文再回答；微调管风格与路子，RAG 管出处。
- **给别人用**：GGUF 传到 Hugging Face，套个 Gradio 界面开 Spaces，别人打开链接就能聊；或用 Ollama 本地跑。
- **版本说明**：按 Unsloth 2026 年初的常用接口写，未在 Kaggle 上实跑；若 `SFTTrainer` 的 `dataset_text_field` 参数在新版 trl 里报错，按报错提示改用 `formatting_func` 方式传入同一批 `texts`。